# ⚙️ Generating Embeddings with Gemini

**Day 5 — Notebook 2 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- The Gemini embedding models and their capabilities
- Task types: how telling the model your use case improves quality
- Batch embedding for efficiency
- Handling long texts and chunking strategies
- Practical patterns for storing and reusing embeddings

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" numpy

In [ ]:
import sys, os, json, time
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

from google.genai import errors

try:
    client = genai.Client(api_key=get_api_key())
except errors.APIError as e:
    print(f"Gemini API Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
else:
    MODEL_ID = "gemini-2.5-flash"
    print("✅ Ready!")
EMBEDDING_MODEL = "gemini-embedding-001"

---

## 1. Gemini Embedding Models

Google provides dedicated embedding models optimised for semantic tasks:

| Model | Dimensions | Max Input Tokens | Best For |
|-------|-----------|-----------------|----------|
| `gemini-embedding-001` | 3072 | 8,192 | General purpose, retrieval, clustering |
| `gemini-embedding-001` | 3072 | 8,192 | General purpose, retrieval, clustering, similarity |

For most tasks `gemini-embedding-001` is the right choice — it's fast, affordable, and produces high-quality 3072-dimensional embeddings.

In [ ]:
# Basic single embedding
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="Artificial intelligence is changing the world.",
)

vec = result.embeddings[0].values
print(f"Model: {EMBEDDING_MODEL}")
print(f"Embedding dimensions: {len(vec)}")
print(f"Sample values (first 5): {[round(v, 6) for v in vec[:5]]}")

---

## 2. Task Types: Optimising for Your Use Case

The `gemini-embedding-001` model supports **task types** — hints that tell the model how the embedding will be used. This significantly improves quality.

| Task Type | Use When |
|-----------|----------|
| `RETRIEVAL_DOCUMENT` | Embedding documents to be searched/retrieved |
| `RETRIEVAL_QUERY` | Embedding a search query |
| `SEMANTIC_SIMILARITY` | Comparing two texts for similarity |
| `CLASSIFICATION` | Classifying text into categories |
| `CLUSTERING` | Grouping texts by topic |

In [ ]:
def embed_document(text: str) -> np.ndarray:
    """Embed a document intended to be retrieved."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT",
        ),
    )
    return np.array(result.embeddings[0].values)


def embed_query(text: str) -> np.ndarray:
    """Embed a search query."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
        ),
    )
    return np.array(result.embeddings[0].values)


# Test: a query vs a document — the right task type improves retrieval accuracy
query = "How does photosynthesis work?"
document = "Photosynthesis is the process by which plants convert sunlight into glucose using CO2 and water, releasing oxygen as a byproduct."

q_vec = embed_query(query)
d_vec = embed_document(document)

similarity = np.dot(q_vec, d_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(d_vec))
print(f"Query: '{query}'")
print(f"Document: '{document[:80]}...'")
print(f"Similarity (with task types): {similarity:.4f}")

---

## 3. Batch Embedding (Efficient API Usage)

Instead of making one API call per text, you can embed **multiple texts in a single call**. This is much more efficient.

In [ ]:
documents = [
    "The Amazon rainforest produces 20% of the world's oxygen.",
    "Mount Everest is the highest peak on Earth at 8,848 metres.",
    "The Great Wall of China stretches over 21,000 kilometres.",
    "The Pacific Ocean is the largest and deepest ocean on Earth.",
    "Antarctica is the coldest, driest, and windiest continent.",
]

# Batch embedding — single API call for all documents
start = time.time()
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=documents,  # Pass a list!
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
    ),
)
elapsed = time.time() - start

embeddings = [np.array(e.values) for e in result.embeddings]

print(f"Embedded {len(documents)} documents in {elapsed:.2f}s (single API call)")
print(f"Shape of each embedding: ({len(embeddings[0])},)")
print(f"\nFirst document: '{documents[0][:60]}'")
print(f"First 5 values: {[round(v, 4) for v in embeddings[0][:5]]}")

---

## 4. Handling Long Texts

`gemini-embedding-001` has a **8,192 token limit**. For longer documents (like full articles or PDFs), you need to chunk the text first.

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """
    Split text into overlapping chunks by word count.
    Overlap ensures context isn't lost at chunk boundaries.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# Simulate a long article
long_article = """The history of artificial intelligence dates back to the mid-20th century. 
Alan Turing proposed the concept of a machine that could simulate any human intelligence in 1950. 
The term 'artificial intelligence' was coined by John McCarthy in 1956 at the Dartmouth Conference. 
Early AI research focused on symbolic reasoning and logic-based systems. 
These systems were brittle and struggled with real-world ambiguity. 
The AI winters of the 1970s and 1980s saw funding and interest dramatically reduced. 
The resurgence of neural networks in the 1980s, pioneered by researchers like Geoffrey Hinton, 
laid the groundwork for modern deep learning. 
The introduction of backpropagation allowed neural networks to be trained effectively. 
The 2012 ImageNet competition was a watershed moment — a deep neural network (AlexNet) 
dramatically outperformed traditional computer vision methods. 
This sparked the modern deep learning revolution. 
Transformer architecture, introduced in the 2017 paper 'Attention is All You Need', 
became the foundation for all modern large language models including GPT, BERT, and Gemini.
Today, LLMs are used in everything from search engines to code generation to medical diagnosis."""

chunks = chunk_text(long_article, chunk_size=50, overlap=10)
print(f"Article word count: {len(long_article.split())}")
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1} ({len(chunk.split())} words):")
    print(f"  {chunk[:100]}...")

In [ ]:
# Embed all chunks
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=chunks,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
chunk_embeddings = [np.array(e.values) for e in result.embeddings]
print(f"Embedded {len(chunk_embeddings)} chunks. Each is {len(chunk_embeddings[0])}d.")

---

## 5. Saving and Loading Embeddings

Embeddings are expensive to compute. In production, you should **cache them** rather than re-computing on every request.

In [ ]:
import numpy as np
import json

# Saving embeddings to disk (numpy format — efficient for large sets)
embedding_data = {
    "model": EMBEDDING_MODEL,
    "documents": documents,
    "vectors": [e.tolist() for e in embeddings],
}

with open("embeddings_cache.json", "w") as f:
    json.dump(embedding_data, f)

print("✅ Saved embeddings to embeddings_cache.json")

# Loading back
with open("embeddings_cache.json") as f:
    loaded = json.load(f)

loaded_vectors = [np.array(v) for v in loaded["vectors"]]
print(f"✅ Loaded {len(loaded_vectors)} embeddings from cache")
print(f"Dimensions: {len(loaded_vectors[0])}")

---

## 6. Embedding Titles vs. Full Text

You can also provide a **title** alongside the content to improve retrieval quality.

In [ ]:
# Embedding with title (useful for articles, docs with clear headings)
result_with_title = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="Photosynthesis is the process by which plants produce food from sunlight.",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        title="How Plants Make Food",  # Title provides additional context
    ),
)

result_without_title = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="Photosynthesis is the process by which plants produce food from sunlight.",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
    ),
)

vec_with = np.array(result_with_title.embeddings[0].values)
vec_without = np.array(result_without_title.embeddings[0].values)

# Check how different the two embeddings are
diff = np.linalg.norm(vec_with - vec_without)
print(f"L2 distance between embeddings with vs. without title: {diff:.4f}")
print("(Non-zero distance shows the title influences the embedding)")

---

## 🏋️ Exercise 1: Task Type Comparison

Compare how using the correct task type affects retrieval quality.

In [ ]:
# Exercise 1: Does task type actually matter?
# TODO: Run this experiment:
# 1. Embed the same query twice: once with RETRIEVAL_QUERY, once with SEMANTIC_SIMILARITY
# 2. Embed 3 documents twice: once with RETRIEVAL_DOCUMENT, once with SEMANTIC_SIMILARITY
# 3. Compare the similarity scores — does the "correct" task type give higher scores
#    for the most relevant document?

query = "What causes thunderstorms?"

doc_relevant = "Thunderstorms form when warm, moist air rises rapidly, causing electrical charges to build up in clouds."
doc_somewhat = "Rain is a form of precipitation that falls from clouds."
doc_unrelated = "The stock market saw heavy trading volume last Tuesday."

# TODO: Embed query + docs with both task types and compare
# Print a comparison table

---

## 🏋️ Exercise 2: Build a Reusable Embedding Cache

Build a class that caches embeddings to avoid re-computing them.

In [ ]:
# Exercise 2: Embedding cache
# TODO: Complete the EmbeddingCache class

class EmbeddingCache:
    """
    Caches embeddings so the same text is never embedded twice.
    Falls back to calling the API on cache miss.
    """
    
    def __init__(self, model: str = EMBEDDING_MODEL):
        self.model = model
        self._cache = {}  # text → numpy array
        self.hits = 0
        self.misses = 0
    
    def get(self, text: str, task_type: str = "RETRIEVAL_DOCUMENT") -> np.ndarray:
        cache_key = f"{task_type}::{text}"
        
        if cache_key in self._cache:
            self.hits += 1
            # TODO: Return cached embedding
            pass
        else:
            self.misses += 1
            # TODO: Call the API, store in cache, return result
            pass
    
    def stats(self):
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        print(f"Cache stats: {self.hits} hits, {self.misses} misses ({hit_rate:.0%} hit rate)")


# Test it
# cache = EmbeddingCache()
# texts = ["Hello world", "Hello world", "How are you?", "Hello world"]
# for t in texts:
#     vec = cache.get(t)
#     print(f"Got embedding for: '{t}'")
# cache.stats()

---

## Key Takeaways

| Concept | Best Practice |
|---------|---------------|
| Task types | Always specify `task_type` — it meaningfully improves quality |
| Batch requests | Embed multiple texts in one call to reduce latency and cost |
| Long texts | Chunk to stay within the token limit (8,192 tokens) |
| Caching | Store embeddings — they don't change for the same text |
| Titles | Use `title` parameter for documents with clear headings |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Gemini Embedding Models | Docs | [ai.google.dev/gemini-api/docs/models/gemini#text-embedding](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding) |
| Embeddings Guide | Docs | [ai.google.dev/gemini-api/docs/embeddings](https://ai.google.dev/gemini-api/docs/embeddings) |

---

**Next up:** [03_Similarity_Search.ipynb](./03_Similarity_Search.ipynb)